# Train the CUB-200 models on a free Colab GPU

`Runtime > Change runtime type > T4 GPU`, then run the cells in order. Every step is a project CLI command;
this notebook only sets up the environment (spec 5.4: notebooks are not the main code).

In [ ]:
!git clone https://github.com/Youssefmohd58/deep_learning.git
%cd deep_learning
!git checkout restructure-milestone1   # remove this line once the branch is merged into main
!pip install -q -r requirements.txt
!python -m src.cli check

## Keep results on Google Drive
Checkpoints and outputs are written to Drive so a disconnect does not lose them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cub200_project'
!mkdir -p {DRIVE}/checkpoints {DRIVE}/outputs
!rm -rf checkpoints outputs && ln -s {DRIVE}/checkpoints checkpoints && ln -s {DRIVE}/outputs outputs

## 1. Hyperparameter tuning (grid + random search)

In [ ]:
!python -m src.cli tune --config configs/tuning_scratch_cnn.yaml --set data.num_workers=2
!python -m src.cli tune --config configs/tuning_resnet18.yaml --set data.num_workers=2

## 2. Final training at 224px with the tuned settings

In [ ]:
!python -m src.cli train --config configs/scratch_cnn.yaml --use-tuned --set data.num_workers=2
!python -m src.cli train --config configs/resnet18.yaml --use-tuned --set data.num_workers=2
!python -m src.cli train --config configs/efficientnet_b0.yaml --set data.num_workers=2

## 3. Evaluate, compare, benchmark

In [ ]:
!python -m src.cli evaluate --model all --set data.num_workers=2
!python -m src.cli compare
!python -m src.cli benchmark

## 4. Experiment tracking (MLflow)
All runs are logged to `outputs/mlflow.db` (+ `outputs/mlartifacts/`) on your Drive. Copy the `outputs/`
folder into the project on your own computer and run
`mlflow ui --backend-store-uri sqlite:///outputs/mlflow.db` to browse and compare runs (screenshots for the report).